# ⚡ Módulo 13 - Notebook 02: Consultas Declarativas PySpark SQL

## 📊 SQL Distribuido y Vistas Temporales

**Libro:** Saliendo de lo Pandito  
**Módulo:** 13 - PySpark SQL Window DeltaLake  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Escribir** queries SQL complejas en PySpark  
✅ **Crear** y gestionar vistas temporales  
✅ **Combinar** SQL con DataFrame API  
✅ **Optimizar** queries con CTEs y subqueries  
✅ **Trabajar** con tablas del catálogo

---

## 📋 Pre-requisitos

* ✅ Notebook 13_01 completado (Introducción a PySpark SQL)
* ✅ Conocimiento de SQL estándar (SELECT, JOIN, GROUP BY)
* ✅ Familiaridad con Spark DataFrames

---

## 📚 Contenido

1. Vistas Temporales (Temp Views)
2. CTEs (Common Table Expressions)
3. Subqueries y Nested Queries
4. Catálogo de Tablas
5. SQL vs DataFrame API
6. Caso Integrador: Análisis SQL Completo

---

## 💡 Por qué importa

**SQL distribuido democratiza Big Data:**

* 👥 **Accesible:** Analistas sin programación
* 📊 **Expresivo:** Queries complejas en SQL puro
* ⚡ **Optimizado:** Catalyst mejora tu SQL
* 🔄 **Integrado:** Combina con Python cuando necesites

**SQL + Spark = Data warehouses modernos**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    # Registrar como vista temporal GLOBAL (disponible en todas las sesiones)
    df_ventas.createOrReplaceGlobalTempView("ventas_global")
    
    # Registrar como vista temporal LOCAL (solo esta sesión)
    df_ventas.createOrReplaceTempView("ventas")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df_ventas.count():,}")
    print(f"   🗃️ Particiones: {df_ventas.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    print(f"\n📋 Vistas SQL creadas:")
    print(f"   • LOCAL:  'ventas' (solo esta sesión)")
    print(f"   • GLOBAL: 'global_temp.ventas_global' (todas las sesiones)")
    
    # Listar todas las tablas del catálogo
    print(f"\n📚 Tablas disponibles en el catálogo:")
    spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)
    
    print(f"\n🎯 Este notebook ejecutará SQL PURO sobre datos REALES")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 SQL Declarativo: El Poder de la Simplicidad

### 📋 Vistas Temporales

**Vista Temporal:** Un alias SQL para un DataFrame.

**Tipos:**

**1️⃣ Vista LOCAL (una sesión):**
```python
df.createOrReplaceTempView("mi_vista")
# Uso: SELECT * FROM mi_vista
```

**2️⃣ Vista GLOBAL (todas las sesiones):**
```python
df.createOrReplaceGlobalTempView("mi_vista_global")
# Uso: SELECT * FROM global_temp.mi_vista_global
```

**Diferencia:**
* LOCAL: Desaparece al cerrar sesión
* GLOBAL: Persiste entre sesiones (útil en clusters)

---

### 🔗 CTEs (Common Table Expressions)

**CTE:** Subquery nombrado (más legible que subqueries anidados).

**Sintaxis:**
```sql
WITH ventas_altas AS (
    SELECT zona, SUM(ventas) as total
    FROM ventas
    WHERE ventas > 100000
    GROUP BY zona
)
SELECT *
FROM ventas_altas
WHERE total > 1000000
ORDER BY total DESC
```

**Ventajas:**
* ✅ Más legible que subqueries anidados
* ✅ Reutilizable en el mismo query
* ✅ Modular y fácil de debug

---

### 📊 Subqueries

**Subquery en WHERE:**
```sql
SELECT *
FROM ventas
WHERE ventas > (SELECT AVG(ventas) FROM ventas)
```

**Subquery en FROM:**
```sql
SELECT zona, AVG(total)
FROM (
    SELECT zona, SUM(ventas) as total
    FROM ventas
    GROUP BY zona, fecha
)
GROUP BY zona
```

---

### 📚 Catálogo de Tablas

**Listar tablas:**
```sql
SHOW TABLES IN catalog.schema
```

**Describir tabla:**
```sql
DESCRIBE EXTENDED catalog.schema.tabla
```

**Ver esquema:**
```python
df = spark.table("catalog.schema.tabla")
df.printSchema()
```

---

### 🔄 SQL vs DataFrame API

**Mismo resultado, diferente sintaxis:**

**SQL:**
```python
result = spark.sql("""
    SELECT 
        zona,
        COUNT(*) as transacciones,
        SUM(ventas) as total
    FROM ventas
    WHERE ventas > 50000
    GROUP BY zona
    HAVING SUM(ventas) > 1000000
    ORDER BY total DESC
""")
```

**DataFrame API:**
```python
result = df \
    .filter(F.col("ventas") > 50000) \
    .groupBy("zona") \
    .agg(
        F.count("*").alias("transacciones"),
        F.sum("ventas").alias("total")
    ) \
    .filter(F.col("total") > 1000000) \
    .orderBy(F.desc("total"))
```

**Ejecución:** IDÉNTICA (mismo plan Catalyst)

---

### 🎯 Cuándo Usar Cada Uno

**Usar SQL cuando:**
* ✅ Queries exploratorias ad-hoc
* ✅ Equipo con conocimiento SQL fuerte
* ✅ Migrando desde SQL tradicional

**Usar DataFrame API cuando:**
* ✅ Lógica compleja en Python
* ✅ Integración con ML/ETL
* ✅ Testing y code reuse

**Mejor práctica:** Combinar ambos según necesidad.

---

### 💡 Performance: Ambos Iguales

```python
# Ambos generan EL MISMO plan de ejecución:
spark.sql("SELECT * FROM ventas WHERE zona = 'Centro'").explain()
df.filter("zona = 'Centro'").explain()

# Resultado: Identical physical plan
```

**Conclusión:** Usa el que prefieras, Spark optimiza ambos igual.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("📊 CONSULTAS DECLARATIVAS PYSPARK SQL")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • createOrReplaceTempView() - Vistas locales")
print("  • createOrReplaceGlobalTempView() - Vistas globales")
print("  • CTEs (WITH) - Common Table Expressions")
print("  • Subqueries en WHERE y FROM")
print("  • Catálogo de tablas")

print("\n📖 Métodos clave:")
print("  - spark.sql('SELECT ...')")
print("  - df.createOrReplaceTempView('nombre')")
print("  - spark.sql('WITH cte AS (...) SELECT ...')")
print("  - spark.catalog.listTables()")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')